## GitHub users by location

**Strategy: month-by-month.** GitHub's Search API returns **at most 1,000 results per search query**. To get as many users as possible, we fetch **every month** separately (e.g. `created:2023-01-01..2023-01-31`). Each month is a smaller time window, so we're less likely to hit the 1,000 cap and we can get up to 12×1000 = 12,000 users per year when a location is very active.

**Limitations:** Users who don't set "Location" or use a different spelling won't be included. There is no way to get a complete list of all GitHub users in a country via the public API.

In [ ]:
# Optional: set your token here for this session only (kernel often doesn't see terminal env).
# Run this cell, then re-run the fetch cell below.
# GITHUB_TOKEN = "ghp_yourActualTokenHere"

In [5]:
import calendar
import requests
import csv
import time
import os
from datetime import datetime

# --- CONFIGURATION ---
# Token: env GITHUB_TOKEN / GITHUB_KEY / GH_TOKEN, or .env file (no extra package needed)
def _load_dotenv():
    path = os.path.join(os.getcwd(), ".env")
    if not os.path.isfile(path):
        return
    try:
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line and not line.startswith("#") and "=" in line:
                    k, _, v = line.partition("=")
                    k, v = k.strip(), v.strip().strip("'\"")
                    if k and v and k not in os.environ:
                        os.environ[k] = v
    except Exception:
        pass
_load_dotenv()
_env_token = (os.environ.get("GITHUB_TOKEN") or os.environ.get("GITHUB_KEY") or os.environ.get("GH_TOKEN") or "").strip()
GITHUB_TOKEN = _env_token or "YOUR_GITHUB_TOKEN_HERE"
LOCATION = "Karachi"
OUTPUT_FILE = "github_users_karachi_full.csv"
# Fetch month-by-month from START_YEAR through END_YEAR (inclusive)
START_YEAR = 2010
END_YEAR = 2026
# ---------------------

def fetch_users_for_query(headers, date_query, year_label):
    """Fetch up to 1000 users for a single search query (one date range). Returns (list of user dicts, total_count from API)."""
    users = []
    total_count = 0
    page = 1
    while True:
        url = f"https://api.github.com/search/users?q=location:{LOCATION}+{date_query}&page={page}&per_page=100"
        response = requests.get(url, headers=headers)
        if response.status_code == 403:
            return users, total_count, "rate_limit", ""
        if response.status_code != 200:
            err_detail = response.text[:300] if response.text else ""
            return users, total_count, f"error_{response.status_code}", err_detail
        data = response.json()
        total_count = data.get("total_count", 0)
        items = data.get("items", [])
        if not items:
            break
        for item in items:
            users.append({
                "Username": item["login"],
                "Profile URL": item["html_url"],
                "GitHub ID": item["id"],
                "Year_Group": year_label,
            })
        if page >= 10 or "next" not in response.links:
            break
        page += 1
        time.sleep(2)
    return users, total_count, "ok", ""

def check_token():
    """Verify token works. Returns (True, None) or (False, error_message)."""
    token = GITHUB_TOKEN.strip() if GITHUB_TOKEN else ""
    if not token or token == "YOUR_GITHUB_TOKEN_HERE":
        in_env = "GITHUB_TOKEN" in os.environ or "GH_TOKEN" in os.environ
        hint = "The notebook kernel often doesn't see env vars set in a terminal. "
        if in_env:
            hint += "Env var is set but empty in this process."
        else:
            hint += "Set it in this session: run in a cell: GITHUB_TOKEN = 'ghp_your_token', then re-run. Or add a .env file with GITHUB_TOKEN=ghp_... (pip install python-dotenv)."
        return False, "GITHUB_TOKEN is not set or empty. " + hint
    headers = {"Authorization": f"Bearer {GITHUB_TOKEN}", "Accept": "application/vnd.github.v3+json"}
    r = requests.get("https://api.github.com/user", headers=headers, timeout=10)
    if r.status_code == 401:
        try:
            msg = r.json().get("message", r.text[:200])
        except Exception:
            msg = r.text[:200]
        return False, f"GitHub returned 401 Unauthorized: {msg}"
    if r.status_code != 200:
        return False, f"GitHub returned {r.status_code}: {r.text[:200]}"
    return True, None

def fetch_users():
    headers = {
        "Authorization": f"Bearer {GITHUB_TOKEN}",
        "Accept": "application/vnd.github.v3+json",
    }
    ok, err = check_token()
    if not ok:
        print(err)
        return []

    all_users = []
    seen_logins = set()
    auth_error_shown = False

    for year in range(START_YEAR, END_YEAR + 1):
        print(f"--- {LOCATION} / year {year} (month-by-month) ---")
        for month in range(1, 13):
            start = f"{year}-{month:02d}-01"
            last_day = calendar.monthrange(year, month)[1]
            end = f"{year}-{month:02d}-{last_day:02d}"
            date_query = f"created:{start}..{end}"
            label = f"{year}-{month:02d}"

            users, total_count, status, err_detail = fetch_users_for_query(headers, date_query, label)
            if status == "rate_limit":
                print("  Rate limit — sleeping 60s...")
                time.sleep(60)
                users, total_count, status, err_detail = fetch_users_for_query(headers, date_query, label)
            if status == "error_401" and not auth_error_shown:
                auth_error_shown = True
                print("  401 Unauthorized. Token may be missing, wrong, or expired.")
                if err_detail:
                    print("  GitHub says:", err_detail[:500])
                return all_users
            if status.startswith("error"):
                print(f"  {label}: {status}")
                continue

            new_count = 0
            for rec in users:
                if rec["Username"] not in seen_logins:
                    seen_logins.add(rec["Username"])
                    all_users.append(rec)
                    new_count += 1
            if users or total_count:
                print(f"  {label}: {len(users)} users (total_count={total_count}), +{new_count} new → total {len(all_users)}")
            time.sleep(2)

    return all_users

def save_to_csv(data):
    if not data: return
    with open(OUTPUT_FILE, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=data[0].keys())
        writer.writeheader()
        writer.writerows(data)
    print(f"\nDone! Saved {len(data)} unique users to {OUTPUT_FILE}")

if __name__ == "__main__":
    results = fetch_users()
    save_to_csv(results)

--- Karachi / year 2010 (month-by-month) ---
  2010-01: 5 users (total_count=5), +5 new → total 5
  2010-02: 5 users (total_count=5), +5 new → total 10
  2010-03: 1 users (total_count=1), +1 new → total 11
  2010-04: 6 users (total_count=6), +6 new → total 17
  2010-05: 2 users (total_count=2), +2 new → total 19
  2010-06: 2 users (total_count=2), +2 new → total 21
  2010-07: 7 users (total_count=7), +7 new → total 28
  2010-08: 2 users (total_count=2), +2 new → total 30
  2010-09: 2 users (total_count=2), +2 new → total 32
  2010-10: 1 users (total_count=1), +1 new → total 33
  2010-11: 2 users (total_count=2), +2 new → total 35
  2010-12: 4 users (total_count=4), +4 new → total 39
--- Karachi / year 2011 (month-by-month) ---
  2011-01: 4 users (total_count=4), +4 new → total 43
  2011-02: 3 users (total_count=3), +3 new → total 46
  2011-03: 4 users (total_count=4), +4 new → total 50
  2011-04: 1 users (total_count=1), +1 new → total 51
  2011-05: 6 users (total_count=6), +6 new → to

## Enrich one CSV with public repos and lifetime commits

This section takes a **single input CSV** (currently `github_users_islamabad_full.csv`) and, for each `Username`:

1. Uses the **REST API** (`/users/{login}`) to fetch the user's **public repository count**, stored in the `Public_Repositories` column.
2. Uses the **GraphQL API** to iterate over all of the user's **non‑fork repositories that they own**, walks each repository's **default branch commit history**, and sums the `history.totalCount` values to approximate that user's **lifetime number of commits** across their own repos (`Lifetime_Commits`).
3. Runs the REST and GraphQL calls **concurrently per user** using `asyncio` + `aiohttp`, and rotates through multiple tokens loaded from `.env` to better handle rate limiting.

The result is a new CSV with the original columns plus **`Public_Repositories`** and **`Lifetime_Commits`** for every user in the chosen file.


In [1]:
import asyncio
import aiohttp
import csv
import time
import json
from itertools import cycle
import os

INPUT_CSV = "github_users_lahore_full.csv"
CHECKPOINT_FILE = "checkpoint.json"  # saves progress here
MAX_CONCURRENT_USERS = 5
REST_URL = "https://api.github.com/users/{}"

token_pool = None
token_lock = None
TOKENS = []

def load_tokens_from_env(env_path=".env"):
    tokens = []
    try:
        with open(env_path, "r") as f:
            for line in f:
                line = line.strip()
                if not line or line.startswith("#"):
                    continue
                if "=" in line:
                    key, _, value = line.partition("=")
                    key = key.strip()
                    value = value.strip()
                    if key.startswith("ghp_token"):
                        tokens.append(value)
    except FileNotFoundError:
        raise FileNotFoundError(f"❌ .env file not found at: {os.path.abspath(env_path)}")
    return tokens

def load_checkpoint():
    """Load previously saved progress. Returns dict of {username: {cols}}."""
    if os.path.exists(CHECKPOINT_FILE):
        with open(CHECKPOINT_FILE, "r") as f:
            data = json.load(f)
            print(f"♻️  Checkpoint found — {len(data):,} users already completed, resuming...")
            return data
    return {}

def save_checkpoint(completed_data: dict):
    """Save progress to checkpoint file."""
    with open(CHECKPOINT_FILE, "w") as f:
        json.dump(completed_data, f)

async def get_next_token():
    async with token_lock:
        try:
            return next(token_pool)
        except StopIteration:
            return TOKENS[0]

async def fetch_user_data(session, username, retry=0):
    if retry > 5:
        return {"Public_Repositories": "N/A"}

    token = await get_next_token()
    headers = {
        "Authorization": f"token {token}",
        "Accept": "application/vnd.github+json",
    }
    try:
        async with session.get(
            REST_URL.format(username.strip()),
            headers=headers,
            timeout=aiohttp.ClientTimeout(total=30)
        ) as resp:
            if resp.status == 200:
                data = await resp.json()
                return {"Public_Repositories": data.get("public_repos", 0)}
            elif resp.status in (429, 403):
                print(f"  ⚠️  Rate limited — waiting 10 minutes...")
                await asyncio.sleep(600)
                return await fetch_user_data(session, username, retry + 1)
            elif resp.status == 404:
                return {"Public_Repositories": "Not Found"}
            else:
                return {"Public_Repositories": "N/A"}
    except asyncio.TimeoutError:
        await asyncio.sleep(2)
        return await fetch_user_data(session, username, retry + 1)
    except Exception:
        return {"Public_Repositories": "N/A"}

async def fetch_lifetime_commits(session, username, retry=0):
    if retry > 5:
        return -1

    GRAPHQL_URL = "https://api.github.com/graphql"
    QUERY = """
    query($login: String!, $cursor: String) {
      user(login: $login) {
        repositories(
          first: 100
          isFork: false
          ownerAffiliations: OWNER
          after: $cursor
        ) {
          pageInfo { hasNextPage endCursor }
          nodes {
            name
            defaultBranchRef {
              target {
                ... on Commit {
                  history { totalCount }
                }
              }
            }
          }
        }
      }
    }
    """
    total = 0
    cursor = None
    token = await get_next_token()
    headers = {"Authorization": f"bearer {token}", "Content-Type": "application/json"}

    while True:
        payload = {"query": QUERY, "variables": {"login": username, "cursor": cursor}}
        try:
            async with session.post(GRAPHQL_URL, json=payload, headers=headers,
                                    timeout=aiohttp.ClientTimeout(total=30)) as resp:
                if resp.status == 200:
                    data = await resp.json()
                    if "errors" in data or not data.get("data", {}).get("user"):
                        return -1
                    repos = data["data"]["user"]["repositories"]
                    for node in repos["nodes"]:
                        ref = node.get("defaultBranchRef")
                        if ref and ref.get("target"):
                            total += ref["target"]["history"]["totalCount"]
                    if repos["pageInfo"]["hasNextPage"]:
                        cursor = repos["pageInfo"]["endCursor"]
                    else:
                        break
                elif resp.status in (429, 403):
                    print(f"  ⚠️  GraphQL rate limited — waiting 10 minutes...")
                    await asyncio.sleep(600)
                else:
                    return -1
        except Exception:
            return -1

    return total

async def process_row(session, row, semaphore, checkpoint, checkpoint_lock):
    async with semaphore:
        username = row["Username"].strip()

        user_data, commits = await asyncio.gather(
            fetch_user_data(session, username),
            fetch_lifetime_commits(session, username)
        )

        row["Public_Repositories"] = user_data.get("Public_Repositories", "N/A")
        row["Lifetime_Commits"] = commits if commits >= 0 else "N/A"

        # Save to checkpoint immediately after each user
        async with checkpoint_lock:
            checkpoint[username] = {
                "Public_Repositories": row["Public_Repositories"],
                "Lifetime_Commits": row["Lifetime_Commits"]
            }

        return row

async def main():
    global token_lock, token_pool, TOKENS

    env_locations = [
        ".env",
        os.path.join(os.getcwd(), ".env"),
        os.path.join(os.path.dirname(os.path.abspath("__file__")), ".env"),
    ]

    TOKENS = []
    for path in env_locations:
        try:
            TOKENS = load_tokens_from_env(path)
            if TOKENS:
                print(f"✅ Loaded {len(TOKENS)} token(s) from: {os.path.abspath(path)}")
                break
        except FileNotFoundError:
            continue

    if not TOKENS:
        print(f"❌ No tokens found!")
        return

    token_pool = cycle(TOKENS)
    token_lock = asyncio.Lock()

    # Load checkpoint (previously completed users)
    checkpoint = load_checkpoint()
    checkpoint_lock = asyncio.Lock()

    rows = []
    with open(INPUT_CSV, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        original_fields = list(reader.fieldnames)
        for row in reader:
            rows.append(row)

    for col in ["Public_Repositories", "Lifetime_Commits"]:
        if col not in original_fields:
            original_fields.append(col)

    # Split into already done vs pending
    pending_rows = []
    skipped = 0
    for row in rows:
        username = row["Username"].strip()
        if username in checkpoint:
            # Fill in saved values, no API call needed
            row["Public_Repositories"] = checkpoint[username]["Public_Repositories"]
            row["Lifetime_Commits"] = checkpoint[username]["Lifetime_Commits"]
            skipped += 1
        else:
            pending_rows.append(row)

    print(f"📂 Loaded {len(rows):,} users — ⏭️  {skipped:,} already done, 🔄 {len(pending_rows):,} remaining\n")

    if not pending_rows:
        print("✅ All users already completed! Writing final CSV...")
    else:
        semaphore = asyncio.Semaphore(MAX_CONCURRENT_USERS)
        start = time.time()
        completed = 0
        checkpoint_save_counter = 0

        connector = aiohttp.TCPConnector(limit=200)
        async with aiohttp.ClientSession(connector=connector) as session:
            indexed_tasks = [
                (i, asyncio.ensure_future(
                    process_row(session, row, semaphore, checkpoint, checkpoint_lock)
                ))
                for i, row in enumerate(pending_rows)
            ]

            for i, task in indexed_tasks:
                result = await task
                pending_rows[i] = result
                completed += 1
                checkpoint_save_counter += 1

                # Save checkpoint to disk every 100 users
                if checkpoint_save_counter >= 100:
                    async with checkpoint_lock:
                        save_checkpoint(checkpoint)
                    checkpoint_save_counter = 0
                    print(f"  💾 Checkpoint saved!")

                if completed % 100 == 0:
                    elapsed = time.time() - start
                    rate = completed / elapsed
                    remaining = (len(pending_rows) - completed) / rate
                    print(f"  [{completed:,}/{len(pending_rows):,}] {rate:.1f} users/sec | ETA: {remaining/60:.1f} min")

        # Final checkpoint save
        save_checkpoint(checkpoint)
        print(f"  💾 Final checkpoint saved!")

    # Merge pending results back into full rows list
    pending_idx = 0
    for row in rows:
        username = row["Username"].strip()
        if username not in checkpoint or \
           "Public_Repositories" not in row:
            row.update(pending_rows[pending_idx])
            pending_idx += 1

    # Write final CSV
    with open(INPUT_CSV, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=original_fields)
        writer.writeheader()
        writer.writerows(rows)

    # Clean up checkpoint file after successful completion
    if os.path.exists(CHECKPOINT_FILE):
        os.remove(CHECKPOINT_FILE)
        print(f"🗑️  Checkpoint file cleaned up")

    print(f"\n✅ Done! '{INPUT_CSV}' updated with both columns")

await main()

✅ Loaded 10 token(s) from: e:\git-city\.env
♻️  Checkpoint found — 26,615 users already completed, resuming...
📂 Loaded 32,560 users — ⏭️  26,615 already done, 🔄 5,945 remaining

  💾 Checkpoint saved!
  [100/5,945] 5.5 users/sec | ETA: 17.7 min
  💾 Checkpoint saved!
  [200/5,945] 5.3 users/sec | ETA: 18.2 min
  💾 Checkpoint saved!
  [300/5,945] 5.3 users/sec | ETA: 17.7 min
  💾 Checkpoint saved!
  [400/5,945] 5.6 users/sec | ETA: 16.6 min
  💾 Checkpoint saved!
  [500/5,945] 5.4 users/sec | ETA: 16.9 min
  💾 Checkpoint saved!
  [600/5,945] 5.4 users/sec | ETA: 16.5 min
  💾 Checkpoint saved!
  [700/5,945] 5.6 users/sec | ETA: 15.7 min
  💾 Checkpoint saved!
  [800/5,945] 5.4 users/sec | ETA: 15.9 min
  💾 Checkpoint saved!
  [900/5,945] 5.4 users/sec | ETA: 15.5 min
  💾 Checkpoint saved!
  [1,000/5,945] 5.5 users/sec | ETA: 14.9 min
  💾 Checkpoint saved!
  [1,100/5,945] 5.6 users/sec | ETA: 14.4 min
  💾 Checkpoint saved!
  [1,200/5,945] 5.6 users/sec | ETA: 14.2 min
  💾 Checkpoint saved!
 